<a href="https://colab.research.google.com/github/Gabriel-Bela/-M-dulo-de-Gerenciamento-de-Pouso-e-Estabiliza-o-de-Base-MGPEB-/blob/main/Base_Aurora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
MGPEB – Protótipo (Aurora Siger)

Inclui:
- Módulos fixos (HAB, ENE, LAB, LOG, MED) com preenchimento por input() (um por vez)
- Regras booleanas (AND/OR/NOT) para autorizar/alertar/esperar
- Fila/listas/pilha + bubble sort (clássico)
- Regra didática: energia da base crítica -> ENE-01 fura fila
- Altura x Tempo: g Marte fixo -> mostra só tempo estimado de pouso
- ESG
- Para testar o pouso de todos os módulos: 5 ciclos principais (fixo)
- Finalização: lista POUSADOS / ESPERA / ALERTA
- Energia da base: só muda quando ENE-01 pousa
"""

# =========================
# CONSTANTES DIDÁTICAS
# =========================
G_MARTE = 3.71
LIMIAR_COMBUSTIVEL = 20
LIMIAR_ENERGIA_BASE = 50.0

# =========================
# 1) ESTRUTURAS + MÓDULOS
# =========================

def criar_modulo(nome, tipo):
    return {
        "nome": nome,
        "tipo": tipo,
        "combustivel": None,
        "massa": None,
        "criticidade": None,
        "eta_orbita_min": None,
        "energia_kwh": None,
        "status": "AGUARDANDO"
    }

fila_principal = []
lista_pousados = []
lista_espera = []
lista_alerta = []
pilha_log = []

def enfileirar(fila, modulo):
    fila.append(modulo)

def desenfileirar(fila):
    if len(fila) == 0:
        return None
    primeiro = fila[0]
    del fila[0]
    return primeiro

def log_evento(msg):
    pilha_log.append(msg)

MODULOS_FIXOS = [
    criar_modulo("HAB-01", "Habitação"),
    criar_modulo("ENE-01", "Energia"),
    criar_modulo("LAB-01", "Laboratório Científico"),
    criar_modulo("LOG-01", "Logística"),
    criar_modulo("MED-01", "Suporte Médico"),
]

def texto_esg():
    return (
        "Módulo de Gerenciamento de Pouso e Estabilização da Base Aurora Siger:\n"
        "Seção: Princípios ESG na Base Aurora\n"
        "Ambiental (E): reduzir poeira/impactos e evitar zonas de alto valor científico.\n"
        "Social (S): segurança e transparência nas decisões (logs e critérios claros).\n"
        "Governança (G): regras auditáveis, revisáveis e decisões justificáveis.\n"
        "Nessa missão temos como objetivo pousar todos os módulos com segurança de acordo com as variáveis e suas prioridades!\n"
    )

# =========================
# INPUTS
# =========================

def ler_int(msg, minimo=None, maximo=None):
    while True:
        txt = input(msg).strip()
        if txt.isdigit():
            v = int(txt)
            if minimo is not None and v < minimo:
                print(f"Valor inválido. Mínimo: {minimo}.")
                continue
            if maximo is not None and v > maximo:
                print(f"Valor inválido. Máximo: {maximo}.")
                continue
            return v
        print("Entrada inválida. Digite um número inteiro.")

def ler_float(msg, minimo=None, maximo=None):
    while True:
        txt = input(msg).strip().replace(",", ".")
        try:
            v = float(txt)
            if minimo is not None and v < minimo:
                print(f"Valor inválido. Mínimo: {minimo}.")
                continue
            if maximo is not None and v > maximo:
                print(f"Valor inválido. Máximo: {maximo}.")
                continue
            return v
        except ValueError:
            print("Entrada inválida. Digite um número (ex.: 12.5).")

def preencher_dados_modulos(modulos):
    print("\n=== PREENCHIMENTO DOS MÓDULOS (um por vez) ===")
    for m in modulos:
        print(f"\n--- {m['nome']} | {m['tipo']} ---")
        m["combustivel"] = ler_float("Combustível (% 0 a 100): ", 0, 100)
        m["massa"] = ler_float("Massa (kg): ", 1, 999999999)
        m["criticidade"] = ler_int("Criticidade (1=mais crítico ... 5=menos crítico): ", 1, 5)
        m["eta_orbita_min"] = ler_int("ETA na órbita (min): ", 0, 999999999)
        m["energia_kwh"] = ler_float("Energia do módulo (kWh): ", 0, 999999999)

    log_evento("Dados preenchidos para todos os módulos pré-definidos.")

# =========================
# 2) PORTAS LÓGICAS (BOOLEANAS)
# =========================

def avaliar_regras(modulo, atmosfera_ok, area_ok, sensores_ok):
    combustivel_ok = (modulo["combustivel"] >= LIMIAR_COMBUSTIVEL)
    autorizar = combustivel_ok and atmosfera_ok and area_ok and sensores_ok
    alerta = (not combustivel_ok) or (not sensores_ok)
    espera = (not atmosfera_ok) or (not area_ok)
    return autorizar, alerta, espera

def ler_condicoes_operacionais():
    print("\nCondições operacionais (1=Sim / 0=Não):")
    atmosfera_ok = (ler_int("Atmosfera OK? ", 0, 1) == 1)
    area_ok = (ler_int("Área de pouso disponível? ", 0, 1) == 1)
    sensores_ok = (ler_int("Sensores OK? ", 0, 1) == 1)
    return atmosfera_ok, area_ok, sensores_ok

# =========================
# 3) ORDENAÇÃO (BUBBLE SORT) + REGRA ENERGIA
# =========================

def bubble_sort_por_criticidade_e_combustivel(fila):
    ordenada = []
    for item in fila:
        ordenada.append(item)

    n = len(ordenada)
    for i in range(n):
        trocou = False
        for j in range(0, n - 1 - i):
            a = ordenada[j]
            b = ordenada[j + 1]

            cond1 = (a["criticidade"] > b["criticidade"])
            cond2 = (a["criticidade"] == b["criticidade"] and a["combustivel"] > b["combustivel"])
            cond3 = (
                a["criticidade"] == b["criticidade"]
                and a["combustivel"] == b["combustivel"]
                and a["eta_orbita_min"] > b["eta_orbita_min"]
            )

            if cond1 or cond2 or cond3:
                ordenada[j], ordenada[j + 1] = ordenada[j + 1], ordenada[j]
                trocou = True

        if not trocou:
            break

    return ordenada

def aplicar_regra_energia_fura_fila(fila, energia_base_kwh):
    if energia_base_kwh >= LIMIAR_ENERGIA_BASE:
        return

    idx = -1
    for i in range(len(fila)):
        if fila[i]["nome"] == "ENE-01":
            idx = i
            break
    if idx == -1:
        return

    ene = fila[idx]
    del fila[idx]
    fila.insert(0, ene)
    log_evento(f"Regra energia crítica: ENE-01 priorizado (energia_base={energia_base_kwh} kWh).")

def reordenar_fila(fila, energia_base_kwh):
    fila[:] = bubble_sort_por_criticidade_e_combustivel(fila)
    log_evento("Fila reordenada por criticidade e combustível (bubble sort).")
    aplicar_regra_energia_fura_fila(fila, energia_base_kwh)

# =========================
# 4) ALTURA x TEMPO (tempo estimado)
# =========================

def tempo_estimado_pouso(h0, v0, g):
    d = (v0 ** 2) + (2 * g * h0)
    if d < 0 or g <= 0:
        return None
    raiz = d ** 0.5
    t = (v0 + raiz) / g
    return t

def calcular_tempo_pouso_por_input():
    print("\n=== TEMPO ESTIMADO PARA POUSO (Altura x Tempo) ===")
    print(f"Gravidade de Marte fixa: g = {G_MARTE} m/s²")
    h0 = ler_float("Altura inicial h0 (m): ", 0.01, 10**9)
    v0 = ler_float("Velocidade vertical inicial v0 (m/s) (use 0 se quiser): ", -10**6, 10**6)

    t = tempo_estimado_pouso(h0, v0, G_MARTE)
    if t is None:
        print("Não foi possível calcular o tempo (valores inválidos).")
        log_evento("Falha no cálculo do tempo de pouso.")
        return None

    print(f"Tempo estimado até o pouso: {t:.2f} s")
    log_evento(f"Tempo de pouso estimado: {t:.2f} s.")
    return t

# =========================
# DECISÃO DE POUSO (reutilizável)
# =========================

def decidir_e_alocar(modulo, atmosfera_ok, area_ok, sensores_ok, energia_base_kwh):
    """
    Aplica as regras booleanas e move o módulo para a lista correta.
    Retorna energia_base_kwh (pode mudar se ENE-01 pousar).
    """
    autorizar, alerta, espera = avaliar_regras(modulo, atmosfera_ok, area_ok, sensores_ok)

    if autorizar:
        modulo["status"] = "POUSADO"
        lista_pousados.append(modulo)
        print("DECISÃO: POUSO AUTORIZADO.")
        log_evento(f"POUSO AUTORIZADO: {modulo['nome']}.")

        # Energia da base só muda quando ENE-01 pousa
        if modulo["nome"] == "ENE-01":
            energia_base_kwh = energia_base_kwh + modulo["energia_kwh"]
            log_evento(f"Energia base após pouso do ENE-01: {energia_base_kwh} kWh.")
        else:
            log_evento("Energia base não alterada (módulo não é ENE-01).")

    elif alerta:
        modulo["status"] = "ALERTA"
        lista_alerta.append(modulo)
        print("DECISÃO: ALERTA.")
        log_evento(f"ALERTA: {modulo['nome']}.")

    elif espera:
        modulo["status"] = "ESPERA"
        lista_espera.append(modulo)
        print("DECISÃO: ESPERA.")
        log_evento(f"ESPERA: {modulo['nome']}.")

    else:
        modulo["status"] = "BLOQUEADO"
        lista_alerta.append(modulo)
        print("DECISÃO: BLOQUEADO (caso residual).")
        log_evento(f"BLOQUEADO: {modulo['nome']}.")

    return energia_base_kwh

# =========================
# CICLOS PRINCIPAIS
# =========================

def simular_ciclo(energia_base_kwh):
    if len(fila_principal) == 0:
        print("\nFila vazia. Nada para processar.")
        log_evento("Ciclo com fila vazia.")
        return energia_base_kwh

    reordenar_fila(fila_principal, energia_base_kwh)
    atual = desenfileirar(fila_principal)

    print(
        f"\nProcessando: {atual['nome']} | {atual['tipo']} | "
        f"crit={atual['criticidade']} | comb={atual['combustivel']}% | energia_mod={atual['energia_kwh']} kWh"
    )
    print(f"Energia base: {energia_base_kwh} kWh (crítico se < {LIMIAR_ENERGIA_BASE} kWh)")

    atmosfera_ok, area_ok, sensores_ok = ler_condicoes_operacionais()
    return decidir_e_alocar(atual, atmosfera_ok, area_ok, sensores_ok, energia_base_kwh)

# =========================
# REPROCESSAMENTO FINAL (ESPERA/ALERTA) com limite de tentativas
# =========================

def reprocessar_pendencias(energia_base_kwh):
    max_tentativas = len(lista_espera) + len(lista_alerta)
    tentativas_feitas = 0

    if max_tentativas == 0:
        return energia_base_kwh

    log_evento(f"Reprocessamento iniciado: limite de tentativas = {max_tentativas}.")

    while (len(lista_espera) > 0 or len(lista_alerta) > 0) and (tentativas_feitas < max_tentativas):
        print("\n=== PENDÊNCIAS DETECTADAS ===")
        print(f"Em ESPERA: {len(lista_espera)} | Em ALERTA: {len(lista_alerta)}")
        print(f"Tentativas restantes: {max_tentativas - tentativas_feitas}")

        opc = ler_int("Tentar reavaliar pouso dessas pendências? (1=Sim / 0=Não): ", 0, 1)
        if opc == 0:
            log_evento("Usuário optou por não reprocessar pendências.")
            break

        escolha = ler_int("Reprocessar qual primeiro? (1=ESPERA / 2=ALERTA): ", 1, 2)

        if escolha == 1 and len(lista_espera) == 0:
            print("Não há módulos em ESPERA. Indo para ALERTA.")
            escolha = 2
        if escolha == 2 and len(lista_alerta) == 0:
            print("Não há módulos em ALERTA. Indo para ESPERA.")
            escolha = 1

        if escolha == 1:
            modulo = lista_espera[0]
            del lista_espera[0]
            origem = "ESPERA"
        else:
            modulo = lista_alerta[0]
            del lista_alerta[0]
            origem = "ALERTA"

        print(f"\nReavaliando (antes em {origem}): {modulo['nome']} | {modulo['tipo']}")

        modulo["status"] = "AGUARDANDO"
        enfileirar(fila_principal, modulo)
        log_evento(f"Pendência reinserida na fila: {modulo['nome']} (origem={origem}).")

        reordenar_fila(fila_principal, energia_base_kwh)
        atual = desenfileirar(fila_principal)

        print(f"\nProcessando novamente: {atual['nome']} | comb={atual['combustivel']}% | crit={atual['criticidade']}")
        print(f"Energia base: {energia_base_kwh} kWh (crítico se < {LIMIAR_ENERGIA_BASE} kWh)")

        atmosfera_ok, area_ok, sensores_ok = ler_condicoes_operacionais()
        energia_base_kwh = decidir_e_alocar(atual, atmosfera_ok, area_ok, sensores_ok, energia_base_kwh)

        tentativas_feitas += 1
        log_evento(f"Tentativa de reprocessamento {tentativas_feitas}/{max_tentativas} concluída.")

    if tentativas_feitas >= max_tentativas and (len(lista_espera) > 0 or len(lista_alerta) > 0):
        print("\nLimite de tentativas atingido. Permanecem pendências sem reavaliação.")
        log_evento("Limite de tentativas atingido no reprocessamento. Pendências remanescentes.")

    return energia_base_kwh

# =========================
# RELATÓRIOS
# =========================

def imprimir_estado():
    print("\n=== ESTADO ATUAL ===")
    print("Fila (aguardando):")
    for m in fila_principal:
        print(
            f" - {m['nome']} | {m['tipo']} | crit={m['criticidade']} | comb={m['combustivel']}% | "
            f"ETA={m['eta_orbita_min']} | energia_mod={m['energia_kwh']} kWh"
        )
    print(f"\nPousados: {len(lista_pousados)} | Em espera: {len(lista_espera)} | Em alerta: {len(lista_alerta)}")

def listar_resultados_finais():
    print("\n=== RESULTADOS FINAIS (LISTAGEM COMPLETA) ===")

    print("\nPOUSADOS (ordem de pouso):")
    if len(lista_pousados) == 0:
        print(" - Nenhum módulo pousado.")
    else:
        for i in range(len(lista_pousados)):
            m = lista_pousados[i]
            print(f" {i+1}. {m['nome']} | {m['tipo']} | crit={m['criticidade']} | comb={m['combustivel']}%")

    print("\nESPERA:")
    if len(lista_espera) == 0:
        print(" - Nenhum módulo em espera.")
    else:
        for m in lista_espera:
            print(f" - {m['nome']} | {m['tipo']} | crit={m['criticidade']} | comb={m['combustivel']}%")

    print("\nALERTA:")
    if len(lista_alerta) == 0:
        print(" - Nenhum módulo em alerta.")
    else:
        for m in lista_alerta:
            print(f" - {m['nome']} | {m['tipo']} | crit={m['criticidade']} | comb={m['combustivel']}%")

def imprimir_logs():
    print("\n=== LOG (PILHA) - mais recente primeiro ===")
    i = len(pilha_log) - 1
    while i >= 0:
        print(" -", pilha_log[i])
        i -= 1

# =========================
# MAIN
# =========================

def main():
    print(texto_esg())
    print("MGPEB - Protótipo com reprocessamento final (ESPERA/ALERTA)")
    print(f"Limiar combustível OK: {LIMIAR_COMBUSTIVEL}%")
    print(f"Limiar energia crítica base: {LIMIAR_ENERGIA_BASE} kWh\n")

    energia_base_kwh = ler_float("Energia atual inicial da base (kWh): ", 0, 999999999)
    log_evento(f"Energia inicial base: {energia_base_kwh} kWh.")

    preencher_dados_modulos(MODULOS_FIXOS)
    for m in MODULOS_FIXOS:
        enfileirar(fila_principal, m)
    log_evento("Módulos enfileirados na fila principal.")

    calcular_tempo_pouso_por_input()

    # Para testar o pouso de todos os módulos: 5 ciclos (fixo)
    ciclos = 5
    for c in range(1, ciclos + 1):
        print(f"\n--- CICLO PRINCIPAL {c} ---")
        energia_base_kwh = simular_ciclo(energia_base_kwh)
        imprimir_estado()

    # Reprocessamento de pendências
    energia_base_kwh = reprocessar_pendencias(energia_base_kwh)

    print("\n=== FINALIZAÇÃO ===")
    imprimir_estado()
    listar_resultados_finais()
    imprimir_logs()

main()
